Complete the exercises below For **Assignment #6**.

Import the following items,
- `pandas as pd`,
- `LinearRegression` from the [`sklearn.linear_model`](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.linear_model) module,
- `make_column_transformer` from [`sklearn.compose`](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.compose),
- `OneHotEncoder` from [`sklearn.preprocessing`](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.preprocessing),
- `make_pipeline` from the [`sklearn.pipeline`](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.pipeline) module, and,
- everything from the [plotnine]() package.

In [ ]:
pip install plotnine

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from plotnine import *


ModuleNotFoundError: No module named 'plotnine'

## Read in our data for this exercise

Use `pd.read_csv` to read in data from the following URL: http://bit.ly/2IgDF0E. Capture the data into a dataframe called `df_voles`.

❗️Hint: just like in `R` we can read data directly from a URL.

In [2]:
url = "http://bit.ly/2IgDF0E"

# Read directly from the URL (same idea as readr::read_csv in R).
# If you are offline, download the CSV and replace `url` with a local filepath.
df_voles = pd.read_csv(url)

df_voles.shape


(56, 4)

Preview the data with the `.head()` method.

The data contains the variables:

- `site` for the id of each random study site (each case or row is a survey/trapping site)
- `voles` for the vole count at each site
- `veg` for the percent cover of vegetation at each site
- `soil` identifying a site as “moist” or “dry”

In [3]:
df_voles.head()


,site,voles,veg,soil
0,1,17,4,moist
1,2,30,33,moist
2,3,54,94,moist
3,4,49,64,moist
4,5,34,32,moist


## EDA

Let's make a few figures from `df_voles` using `ggplot` from **Plotnine**.

In the cell below plot the `voles` variable (y-axis) versus the `veg` variable and color points by the `soil` variable.

In [4]:
(
    ggplot(df_voles, aes(x="veg", y="voles", color="soil"))
    + geom_point(size=2, alpha=0.8)
    + labs(
        x="Percent vegetation cover (veg)",
        y="Vole count (voles)",
        color="Soil",
        title="Voles vs vegetation cover by soil moisture",
    )
    + theme_minimal()
)


NameError: name 'ggplot' is not defined

## Modeling

In the cell below, model `voles` with `soil` and `veg` as predictors in a parallel slopes model. 

Here are the steps I would take:
1. Make a column transformer with `make_column_transformer` that transforms `soil` with `OneHotEncoder(drop="first")` and passes 'veg' through untransformed.
2. Create a pipeline with `make_pipeline` using the column transformer from above and `LinearRegression()` as my model. 
3. Get the `X` (training data) and `y` predictor from `df_voles`
4. Use the `.fit()` method for the pipeline to train the model with `X` and `y`. 

In [5]:
# 1) Column transformer: one-hot encode soil (baseline = first category) and pass veg through.
ct = make_column_transformer(
    (OneHotEncoder(drop="first"), ["soil"]),
    ("passthrough", ["veg"]),
)

# 2) Pipeline: transform -> linear regression
pipe = make_pipeline(ct, LinearRegression())

# 3) X and y
X = df_voles[["soil", "veg"]]
y = df_voles["voles"]

# 4) Fit model
pipe.fit(X, y)

pipe


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('linearregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehotencoder', ...), ('passthrough', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the outpu

Use the function below to get the parameter values for your model from above.

In [6]:
def get_regression_table(pipeline):
    terms=list(pipeline['columntransformer'].get_feature_names_out()) + ['intercept']
    mod = pipeline['linearregression']
    estimates = list(mod.coef_) + [mod.intercept_]
    data = dict(
        term=terms, 
        estimate=estimates,
    )
    return pd.DataFrame(data)

In [7]:
reg_tbl = get_regression_table(pipe)
reg_tbl


,term,estimate
0,onehotencoder__soil_moist,9.100298
1,passthrough__veg,0.259069
2,intercept,15.464026


❓Would protecting a site with high vegetation cover be a more effective way to preserve the vole population than a site with low vegetation cover? Why?

(**Hint:** use your chart above to answer. It's also possible to leverage your regression parameters if you chose to model `voles` with a parallel slopes model.)

**Answer:** Yes. The scatterplot shows vole counts increase as vegetation cover increases, and the fitted parallel-slopes model estimates a positive slope for `veg`. That means, holding soil type fixed, higher vegetation cover predicts higher vole abundance than lower vegetation cover.


In [8]:
beta_veg = float(reg_tbl.loc[reg_tbl["term"].str.contains("veg"), "estimate"].iloc[0])
beta_veg


0.2590689279235672

❓Dry sites typically cost a lot less to purchase and maintain for conservation organizations. Thus, if a conservation organization decides to purchase a few dry sites, roughly what percent cover of vegetation do they need to maintain on these sites (at a minimum) to support a population of about 30 voles at the site?

(**Hint:** In your chart above, draw a line at voles = 30 using `geom_hline` and make a rough estimate for this answer...)

**Answer:** We can estimate this from the plot and/or solve the regression equation for a *dry* site. In the parallel-slopes model, dry is the baseline soil type, so:

`30 ≈ intercept + beta_veg * veg`  →  `veg ≈ (30 - intercept) / beta_veg`

The code cell below computes that implied vegetation cover.


In [9]:
# Plot again with a horizontal reference at 30 voles
plot_30 = (
    ggplot(df_voles, aes(x="veg", y="voles", color="soil"))
    + geom_point(size=2, alpha=0.8)
    + geom_hline(yintercept=30, linetype="dashed")
    + labs(
        x="Percent vegetation cover (veg)",
        y="Vole count (voles)",
        color="Soil",
        title="Voles vs vegetation cover (reference line at 30 voles)",
    )
    + theme_minimal()
)

# Compute the vegetation cover needed for ~30 voles on a DRY site using the fitted model.
intercept = float(reg_tbl.loc[reg_tbl["term"] == "intercept", "estimate"].iloc[0])
veg_needed_dry = (30 - intercept) / beta_veg

plot_30, veg_needed_dry


NameError: name 'ggplot' is not defined

❓The Nature Conservancy is looking at purchasing a site for this species (in the same study area) that has moist soil and 40% vegetation cover. Using the regression equation what would you predict as the possible vole population the site might be able to support?

(**Hint:** Use `.predict(pd.DataFrame({"soil": ["moist"], "veg": [40]}))` with your pipeline.)

**Answer:** The code cell below uses the fitted pipeline to predict the vole count for a moist site with 40% vegetation cover.


In [10]:
pred_moist_40 = float(pipe.predict(pd.DataFrame({"soil": ["moist"], "veg": [40]}))[0])
pred_moist_40


34.92708150903548